## Statistical analysis

Specify the variables to summarize and obtain a p-value for each feature. The original template describes t-tests for p-value calculations; verify the tests used by your installed `clinic_stats` implementation. Summaries can be stratified by `group`, such as train, validation, and test cohorts.

1. Categorical features: counts and proportions.
2. Continuous features: measures of central tendency and dispersion.

In [ ]:
import pandas as pd
import numpy as np
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.utils import print_join_info

task = get_param_in_cwd('task_column') or 'label'
p_value = get_param_in_cwd('p_value') or 0.05
# Set clinic_file to your clinical data file.
test_data = pd.read_csv(get_param_in_cwd('clinic_file'))
stats_columns_settings = get_param_in_cwd('stats_columns')
continuous_columns_settings = get_param_in_cwd('continuous_columns')
mapping_columns_settings = get_param_in_cwd('mapping_columns')
test_data = test_data[[c for c in test_data.columns if c != task]]
test_data['ID'] = test_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
group_info = pd.read_csv(get_param_in_cwd('label_file'))

print_join_info(test_data, group_info)
test_data = pd.merge(test_data, group_info, on='ID', how='inner')
test_data

In [ ]:
test_data['group'].value_counts()

In [ ]:
import math
log_col = ['ALT', 'AST', 'TBil', 'DBil', 'GGT']
for c in log_col:
    test_data[c] = test_data[c].map(lambda x: math.log2(x))
test_data

# Clean feature names

Remove special characters from feature names.

In [ ]:
import re

def map_cnames(x):
    x = re.split('[（|(]', x)[0]
    x = x.replace('-', '_').replace(' ', '_').replace('>', '').replace('/', '_').replace('+', '_')
    return x.strip('_')

test_data.columns = list(map(map_cnames, test_data.columns))
test_data.columns

# Select variables for analysis

Obtain the feature columns to analyze. Detect them automatically if none are specified.

In [ ]:
stats_columns = [c for c in stats_columns_settings or list(test_data.columns[1:-3]) if c not in []]
test_data = test_data.copy()[['ID'] + stats_columns + ['group']]
test_data#['group'].value_counts()

# Feature category mappings

Identify columns requiring category mappings. Infer category ranges when they are not specified.

In [ ]:
mapping_columns = mapping_columns_settings or [c for c in test_data.columns[1:-1] if test_data[c].dtype == object]
mapping_columns

# Encode categorical data

Map nonnumeric values to numerical categories.

In [ ]:
from onekey_algo.custom.utils import map2numerical

data, mapping = map2numerical(test_data, mapping_columns=mapping_columns)
mapping

In [ ]:
data.dtypes

# Continuous variables

Automatically identify potential continuous variables. A column is treated as continuous if its values are noninteger or it has more than five distinct values.

In [ ]:
from onekey_algo.custom.components.comp1 import fillna

test_data = fillna(test_data, fill_mod='50%')
continuous_columns = []
for col in test_data.columns:
    if test_data[col].apply(lambda x: x.is_integer() if isinstance(x, float) else False).all():
        test_data[col] = test_data[col].astype(int)

for c in stats_columns:
#     print(c, np.unique(test_data[c]), test_data[c].dtype)
    if len(np.unique(test_data[c])) > 8 or not np.int8 <= test_data[c].dtype <= np.int64:
        continuous_columns.append(c)
        
continuous_columns = continuous_columns_settings or continuous_columns
continuous_columns = [c for c in continuous_columns if c not in ('')]
continuous_columns

# Impute missing values

In [ ]:
import os
os.makedirs('data', exist_ok=True)
data = test_data
data.to_csv('data/clinical.csv', index=False)
data

### Statistical summaries

`pretty=True` returns a formatted table; `pretty=False` returns a dictionary.

```python
def clinic_stats(data: DataFrame, stats_columns: Union[str, List[str]], label_column='label',
                 group_column: str = None, continuous_columns: Union[str, List[str]] = None,
                 pretty: bool = True) -> Union[dict, DataFrame]:
    """Compute clinical summaries.

    Args:
        data: Input data.
        stats_columns: Columns to summarize.
        label_column: Binary outcome column; defaults to 'label'.
        group_column: Cohort column, e.g., training, test, or validation.
        continuous_columns: Continuous variables to summarize.
        pretty: Whether to format the results as a table.

    Returns:
        A DataFrame or dictionary of statistics.
    """
```

In [ ]:
from onekey_algo.custom.components.stats import clinic_stats

pd.set_option('display.max_rows', None)
stats_train_val = clinic_stats(data[data['group'].isin(['train', 'test'])], 
                               stats_columns= stats_columns,
                               label_column='group', 
                               group_column=None, 
                               continuous_columns= continuous_columns, 
                               pretty=True, verbose=False).reset_index(drop=True)

# stats = clinic_stats(data, 
#                      stats_columns= stats_columns,
#                      label_column='group', 
#                      group_column=None, 
#                      continuous_columns= continuous_columns, 
#                      pretty=True, verbose=False)
# # display(stats)
# stats_train_val['test'] = stats['-label=test']
# stats_train_val['test2'] = stats['-label=test2']
stats_train_val.to_csv('data/stats_all.csv', index=False, encoding='utf_8_sig')
stats_train_val